# Knee MRI: reproducible CoAtNet

**Development version — no leaderboard result claimed.**

One model reads MRI slices and predicts twelve knee findings. It uses separate
attention weights for each finding: the network can focus on different slices
for a meniscus tear and for arthritis.

The proposed improvement is simple: train with the median of three published
sets of report-derived scores. A controlled comparison is running; this is not
yet a demonstrated improvement. The public release will include its actual
score, runtime and baseline-versus-change table.


## Run

Attach the official competition and a **verified model package** containing
weights.pt, model.py, common.py, geometry.py and infer.py. Select GPU, keep internet
off, then Run All. The development path below must be set to your package.
The script decodes the current test DICOM and writes submission.csv. It does not
submit automatically. This notebook is not ready for publication until its model
and Kaggle execution have been verified.


In [ ]:
from pathlib import Path
COMPETITION = Path('/kaggle/input/rsna-knee-abnormality-detection')
MODEL_PACKAGE = Path('/kaggle/input/YOUR-VERIFIED-MODEL-PACKAGE')
OUTPUT = Path('/kaggle/working/submission.csv')


In [ ]:
import sys
required = ['weights.pt', 'model.py', 'common.py', 'geometry.py', 'infer.py']
missing = [name for name in required if not (MODEL_PACKAGE/name).is_file()]
if missing:
    raise FileNotFoundError(f'Attach the verified model package; missing: {missing}')
sys.path.insert(0, str(MODEL_PACKAGE))
from infer import run
submission = run(COMPETITION, MODEL_PACKAGE/'weights.pt', OUTPUT, device='cuda')
print(f'Wrote {len(submission)} studies to {OUTPUT}')
display(submission.head())


## What was measured

The initial comparison uses the same study folds, ImageNet initialization,
224×224 images, eighteen three-slice windows and training budget. Only training
scores differ. AUC measures how well predictions rank positive studies above
negative ones; higher is better. The local reference consists of noisy labels
derived from reports, so it does not establish leaderboard performance.

## Rebuild and caveats

The training scripts and input hashes are included with the final model package.
Create any MRI cache in a private notebook after accepting competition terms.
Derived MRI pixels and reports are not part of the weights package. The compact
cache is lossy and is not a substitute for official DICOM.

Credit: [Raptor training recipe](https://www.kaggle.com/code/dreaddevelopment/knee-mri-training-the-twelve-finding-model),
[Steven Lee geometry](https://www.kaggle.com/code/stevenleehans/rsna-knee-500gb-to-11gib-cpu-pixel-cache),
and timm CoAtNet. See NOTICE.md for modifications, label sources and licensing.
